#Exploratary on Counts Data + Spatial + Weather

# Data Dictionary: Variable Overview

Here is a comprehensive list and explanation of all 37 variables in the dataset, grouped by category:

### 1. Temporal Dimensions (Time)
* **`year`, `month`, `day`, `hour`**: The date and time components of the observation.
* **`time`**: Timestamp string.
* **`interval`**: The duration of the measurement interval (e.g., 15 minutes).
* **`datum_van`**: The start date of the site on use.

### 2. Geographical & Site Information
* **`site_id` / `site_nr`**: Unique identifiers for the monitoring station.
* **`direction`**: Direction of travel (e.g., 'in', 'out').
* **`lat_x`, `lon` / `lat_y`, `long`**: Latitude and longitude coordinates of the site.
* **`naam`**: Name of the monitoring location.
* **`domein`**: The domain or authority managing the site.
* **`wegnr`**: Road number associated with the site.
* **`district`**: Administrative district.
* **`gemeente`**: Municipality name.
* **`description`**: A text description of the site location.

### 3. Traffic Metrics (Target Variables)
* **`count`**: The primary traffic count (e.g., number of cyclists).


### 4. Weather & Environmental Features
* **`temperature_2m`**: Air temperature at 2 meters above ground.
* **`apparent_temperature`**: The perceived "feels like" temperature.
* **`relative_humidity_2m`**: Humidity levels.
* **`precipitation`, `rain`, `snowfall`**: Measures of liquid and solid water falling.
* **`wind_speed_10m`**: Wind speed at 10 meters height.
* **`shortwave_radiation`, `direct_normal_irradiance`**: Solar radiation metrics.
* **`sunshine_duration`**: Total duration of sunlight.

### 5. Spatial & Proximity Features (500m Buffer)
* **`school_count`, `station_count`, `park_count`**: Count of facilities within 500 meters.
* **`dist_nearest_station`, `dist_nearest_school`**: Straight-line distance to the nearest facility.

# Summary of Project Plan & Process
To provide a clear overview of the analysis conducted so far, here is a summary of the completed steps:

1.  **Data Setup & Cleaning**:
    *   Mounted Google Drive and loaded the 2024 Belgian cyclist traffic dataset (2.4M+ records).
    *   Handled missing values in spatial columns and corrected temporal fields to ensure analysis reflects 2024 observations specifically.

2.  **Temporal Analysis**:
    *   Identified hourly bimodal peaks (08:00 and 16:00) on weekdays vs. unimodal recreational peaks on weekends.
    *   Confirmed higher activity on weekdays and a strong seasonal peak in August/September.

3.  **Spatial Analysis**:
    *   Mapped high-traffic municipalities (Hasselt, Leuven, Zemst lead the ranking).
    *   Verified positive correlations between proximity to schools/parks and cyclist volume.

4.  **Weather Analysis**:
    *   Quantified the impact of environmental drivers: Solar radiation and temperature are the strongest positive predictors.
    *   Measured deterrents: Snowfall reduces volume by 46%, while rain and high wind significantly lower activity.

5.  **Baseline Modeling**:
    *   Established an OLS Linear Regression baseline with an R-squared of 0.131.
    *   Performed detailed statistical verification showing that all selected features (Time, Weather, Infrastructure) are highly significant (p-values ≈ 0.000).

# 0 Setup and Load Data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/MDA Assignment/data/final_integrated_data.csv'
# Explicitly set dtypes for columns 26 and 27 to avoid mixed-type warnings
df = pd.read_csv(file_path, dtype={'wegnr': str, 'district': str})

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MDA Assignment/data/final_integrated_data.csv'

In [ ]:
# List all column names in the dataset
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())

# 01 Missing Values Check

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Basic Information and Missing Values
print("### Dataset Info ###")
print(df.info())

print("\n### Missing Values ###")
print(df.isnull().sum())

# 2. Summary Statistics for Numerical Columns
print("\n### Summary Statistics (Numerical) ###")
display(df.describe())

# 3. Distribution of the Target Variable 'count'
plt.figure(figsize=(10, 6))
sns.histplot(df['count'], bins=50, kde=True)
plt.title('Distribution of Cyclist Counts')
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.show()

# 4. Count of unique values for key categorical columns
print("\n### Unique Values in Categorical Columns ###")
cat_cols = ['direction', 'year', 'month', 'district', 'gemeente']
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

Setting these to 0 indicates that no station or school was found within the relevant radius (e.g., 3km).

In [ ]:
import pandas as pd

# Fill missing values for distance columns with 0
# This assumes NA means no station/school was found within the search radius
df['dist_nearest_station'] = df['dist_nearest_station'].fillna(0)
df['dist_nearest_school'] = df['dist_nearest_school'].fillna(0)

# Verify the changes
print("Missing values in dist_nearest_station:", df['dist_nearest_station'].isnull().sum())
print("Missing values in dist_nearest_school:", df['dist_nearest_school'].isnull().sum())

df[['dist_nearest_station', 'dist_nearest_school']].head()

# 02 Counts Data

In [ ]:
# Calculating all temporal features to ensure they refer to 2024 observations
# and not the site installation date (datum_van)

# 1. Create the correct observation date
df['date_dt'] = pd.to_datetime(df[['year', 'month', 'day']])

# 2. Derive day name and day of week (0=Monday)
df['day_name'] = df['date_dt'].dt.day_name()
df['day_of_week'] = df['date_dt'].dt.dayofweek

# 3. Derive weekend status
df['is_weekend'] = df['day_name'].apply(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')

print("Temporal columns synchronized with 2024 observation dates.")
# Quick check: datum_van should be different from date_dt in many cases
display(df[['date_dt',  'day_name', 'is_weekend']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Grouping by hour to find the average cyclist count
# This uses the 'hour' column which is independent of the installation date error
hourly_avg = df.groupby('hour')['count'].mean().reset_index()

# Plotting
plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_avg, x='hour', y='count', marker='o', color='teal')

plt.title('Average Cyclist Count by Hour of Day (Verified 2024 Data)', fontsize=15)
plt.xlabel('Hour (0-23)', fontsize=12)
plt.ylabel('Average Cyclist Count', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by hour and the pre-verified 'is_weekend' status
hourly_trend = df.groupby(['hour', 'is_weekend'])['count'].mean().reset_index()

# Plotting
plt.figure(figsize=(14, 7))
sns.lineplot(data=hourly_trend, x='hour', y='count', hue='is_weekend', marker='o', palette='Set1')

plt.title('Hourly Cyclist Traffic Patterns: Weekdays vs. Weekends (Verified 2024 Data)', fontsize=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Average Cyclist Count', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Day Type')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the correct order of days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Calculate average cyclist count per day of week using the new day_name column
day_stats = df.groupby('day_name')['count'].mean().reindex(day_order).reset_index()

# Plotting
plt.figure(figsize=(12, 6))
sns.barplot(data=day_stats, x='day_name', y='count', palette='viridis', hue='day_name', legend=False)

plt.title('Average Cyclist Count by Day of Week (Monday-Sunday)', fontsize=15)
plt.xlabel('Day of the Week', fontsize=12)
plt.ylabel('Average Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# Display data table
print("Average Counts by Day:")
display(day_stats)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Grouping by month to find the average cyclist count
monthly_avg = df.groupby('month')['count'].mean().reset_index()

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

# 1. Line plot for average counts - changed from bar to line
sns.lineplot(data=monthly_avg, x='month', y='count', marker='o', color='teal', ax=ax1)
ax1.set_title('Average Cyclist Count by Month (Trend)', fontsize=15)
ax1.set_xlabel('Month (1-12)', fontsize=12)
ax1.set_ylabel('Average Count', fontsize=12)
ax1.set_xticks(range(1, 13))
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. Box plot to see distribution per month
sns.boxplot(data=df, x='month', y='count', hue='month', palette='viridis', ax=ax2, showfliers=False, legend=False)
ax2.set_title('Distribution of Cyclist Counts by Month (Excluding Outliers)', fontsize=15)
ax2.set_xlabel('Month (1-12)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.show()

 Temporal Patterns (Verified 2024 Data)
* **Hourly Trends (Weekdays)**: A clear bimodal distribution with peaks at **08:00 AM** and **04:00 PM**, typical of work/school commuting.
* **Hourly Trends (Weekends)**: Commuting peaks disappear, replaced by a single broad peak around **02:00 PM - 03:00 PM**, reflecting recreational use.
* **Weekly Distribution**: Cyclist counts are higher on **weekdays (Mon-Fri)**, peaking on **Thursday**, while they significantly drop during the **weekend**, suggesting the dataset is heavily influenced by utility/commuting cycling.
* **Seasonality**: Strong monthly variation; cycling activity peaks in **August and September** and reaches its minimum in **January and December**.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Check unique values and distribution of the 'direction' column
print("### Direction Column Distribution ###")
direction_counts = df['direction'].value_counts()
print(direction_counts)

# 2. Analyze average cyclist count by direction
direction_impact = df.groupby('direction')['count'].mean().reset_index()
print("\n### Average Cyclist Count by Direction ###")
display(direction_impact)

# 3. Visualization analysis
plt.figure(figsize=(10, 6))
sns.barplot(data=direction_impact, x='direction', y='count', palette='pastel', hue='direction', legend=False)

plt.title('Average Cyclist Traffic by Direction', fontsize=15)
plt.xlabel('Direction of Travel', fontsize=12)
plt.ylabel('Average Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# 4. Advanced analysis: Weekday vs Weekend performance across directions
dir_weekend_impact = df.groupby(['direction', 'is_weekend'])['count'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=dir_weekend_impact, x='direction', y='count', hue='is_weekend', palette='Set2')
plt.title('Directional Traffic: Weekday vs Weekend', fontsize=15)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Box plot to compare 'count' distribution between 'in' and 'out' directions
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='direction', y='count', palette='Set3', hue='direction', showfliers=False, legend=False)

plt.title('Distribution of Cyclist Counts by Direction (Excluding Outliers)', fontsize=15)
plt.xlabel('Direction of Travel', fontsize=12)
plt.ylabel('Cyclist Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

Data Distribution: The dataset is perfectly balanced with exactly 1,209,549 observations for both 'in' and 'out' directions.

Traffic Volume: The 'in' direction shows a slightly higher average cyclist count (11.06) compared to the 'out' direction (10.21).

Temporal Impact: The visualization also confirms that while volumes drop during weekends, the 'in' direction consistently remains busier than 'out' across all day types.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by hour and direction to find average counts
hourly_direction_trend = df.groupby(['hour', 'direction'])['count'].mean().reset_index()

# Plotting
plt.figure(figsize=(14, 7))
sns.lineplot(data=hourly_direction_trend, x='hour', y='count', hue='direction', marker='o', palette='Set1')

plt.title('Hourly Cyclist Traffic: Inbound vs. Outbound', fontsize=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Average Cyclist Count', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Direction')
plt.show()

It reveals a classic commuting pattern: the 'in' direction shows a higher peak during the morning hours (around 08:00), while the 'out' direction follows closely, peaking strongly in the afternoon (around 16:00). Both directions follow a similar bimodal shape, suggesting the routes are used primarily for two-way commuting between residential areas and work or school hubs.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select four different municipalities and one site from each
target_gemeentes = ['Hasselt', 'Leuven', 'Zemst', 'Kortrijk']
selected_sites = {}

for g in target_gemeentes:
    site_id = df[df['gemeente'] == g]['site_id'].iloc[0]
    selected_sites[g] = site_id

# Set up subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=True)
axes = axes.flatten()

for i, (gemeente, site_id) in enumerate(selected_sites.items()):
    site_data = df[df['site_id'] == site_id]
    hourly_data = site_data.groupby(['hour', 'direction'])['count'].mean().reset_index()

    sns.lineplot(data=hourly_data, x='hour', y='count', hue='direction', ax=axes[i], marker='o')
    axes[i].set_title(f'Site {site_id} in {gemeente}', fontsize=14)
    axes[i].set_xlabel('Hour of Day')
    axes[i].set_ylabel('Average Count')
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

Observations from the Site-Specific Analysis:

Commuting Symmetry: Most sites (like Site 137 in Hasselt and Site 69 in Leuven) exhibit a clear inverse relationship: the 'in' direction peaks sharply during the morning commute (08:00), while the 'out' direction peaks during the evening return (16:00).

Volume Variance: There is significant variation in total volume between sites. For instance, Site 69 in Leuven shows much higher peak counts compared to Site 16 in Kortrijk, likely reflecting higher density or more popular cycling infrastructure in those specific areas.

Consistency: Across all four locations, the low traffic levels during the night (22:00 - 05:00) remain consistent, confirming that these sites primarily serve utility or daily transit purposes rather than late-night recreational activity.


# 03 Spatial/Geographical Analysis

Analyzing how cyclist counts vary across different districts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by gemeente and calculate average count
gemeente_avg = df.groupby('gemeente')['count'].mean().sort_values(ascending=False).reset_index()

# Display the top gemeente
print("Average Cyclist Count by Gemeente:")
display(gemeente_avg.head(20)) # Displaying top 20 for brevity

# Visualization
plt.figure(figsize=(12, 10))
sns.barplot(data=gemeente_avg.head(30), y='gemeente', x='count', palette='magma', hue='gemeente', legend=False)
plt.title('Average Cyclist Count per Gemeente (Top 30)', fontsize=15)
plt.xlabel('Average Count', fontsize=12)
plt.ylabel('Gemeente', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import plotly.express as px

# Group by gemeente and calculate average count, also keep lat/lon
# We take the mean of coordinates for each gemeente
map_data = df.groupby('gemeente').agg({
    'count': 'mean',
    'lat_x': 'mean',
    'lon': 'mean'
}).reset_index()

# Create the map
fig = px.scatter_mapbox(
    map_data,
    lat="lat_x",
    lon="lon",
    color="count",
    size="count",
    hover_name="gemeente",
    color_continuous_scale=px.colors.sequential.Viridis,
    size_max=15,
    zoom=7,
    mapbox_style="carto-positron",
    title='Average Cyclist Count by Municipality'
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [ ]:
# Get top 10 municipalities by count
top_10_map_data = map_data.sort_values('count', ascending=False).head(10)

# Create the map for top 10
fig_top_10 = px.scatter_mapbox(
    top_10_map_data,
    lat="lat_x",
    lon="lon",
    color="count",
    size="count",
    hover_name="gemeente",
    color_continuous_scale=px.colors.sequential.Reds,
    size_max=20,
    zoom=7.5,
    mapbox_style="carto-positron",
    title='Top 10 Municipalities by Average Cyclist Count'
)

fig_top_10.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_top_10.show()

# Display table for reference
display(top_10_map_data[['gemeente', 'count']])

Hasselt (39.9), Leuven (35.5), and Zemst (34.2) lead the ranking for the highest average cyclist counts. The interactive map highlights these high-traffic regions, showing a strong concentration of activity in major urban and transit-heavy areas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate average count per gemeente
gemeente_stats = df.groupby('gemeente')['count'].mean().sort_values(ascending=False).reset_index()

# Get top 10 and bottom 10
top_10 = gemeente_stats.head(10).copy()
top_10['Category'] = 'Top 10'

bottom_10 = gemeente_stats.tail(10).copy()
bottom_10['Category'] = 'Bottom 10'

# Combine for plotting
comparison_df = pd.concat([top_10, bottom_10])

# Plotting
plt.figure(figsize=(14, 8))
sns.barplot(data=comparison_df, y='gemeente', x='count', hue='Category', palette={'Top 10': 'seagreen', 'Bottom 10': 'indianred'})

plt.title('Comparison: Top 10 vs. Bottom 10 Municipalities by Average Cyclist Count', fontsize=15)
plt.xlabel('Average Cyclist Count', fontsize=12)
plt.ylabel('Municipality', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.legend(title='Rank Group')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Calculate correlation
corr, p_value = pearsonr(df['park_count'], df['count'])
print(f'Pearson Correlation between park_count and cyclist count: {corr:.4f}')
print(f'P-value: {p_value:.4e}')

# Visualization: Boxplot or Barplot might be clearer since park_count is discrete
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='park_count', y='count', palette='viridis', hue='park_count', legend=False)

plt.title('Average Cyclist Count by Number of Nearby Parks (500m)', fontsize=15)
plt.xlabel('Number of Parks', fontsize=12)
plt.ylabel('Average Cyclist Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

The analysis shows a weak positive correlation of approximately 0.19 between the number of parks within 500m and cyclist counts.

While the correlation is statistically significant (p-value < 0.05 due to the large sample size), the coefficient suggests that park proximity is just one of many factors influencing traffic. The bar chart confirms this trend: locations with a higher number of nearby parks generally show higher average cyclist counts, indicating that green infrastructure likely supports increased cycling activity.

This section compares how cycling patterns shift between workdays and weekends in the top 10 most active municipalities.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate average counts by gemeente and day type
gemeente_behavior = df.groupby(['gemeente', 'is_weekend'])['count'].mean().reset_index()

# 2. To keep the chart readable, we'll focus on the top 10 municipalities by overall traffic
top_10_gemeentes = df.groupby('gemeente')['count'].mean().nlargest(10).index
gemeente_behavior_top = gemeente_behavior[gemeente_behavior['gemeente'].isin(top_10_gemeentes)]

# 3. Plotting
plt.figure(figsize=(14, 8))
sns.barplot(
    data=gemeente_behavior_top,
    y='gemeente',
    x='count',
    hue='is_weekend',
    palette='Set2'
)

plt.title('Top 10 Municipalities: Weekday vs. Weekend Cycling Activity', fontsize=15)
plt.xlabel('Average Cyclist Count', fontsize=12)
plt.ylabel('Municipality', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.legend(title='Day Type')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# List of spatial features to plot
features = ['park_count', 'school_count', 'station_count']
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, feature in enumerate(features):
    sns.scatterplot(data=df, x=feature, y='count', alpha=0.1, s=10, ax=axes[i])
    axes[i].set_title(f'Scatter Plot: {feature} vs Count')
    axes[i].set_xlabel(f'{feature} (500m radius)')
    axes[i].set_ylabel('Cyclist Count')
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

All three spatial features show a dense concentration of points at lower values, which is typical for discrete spatial counts.
While there isn't a strong linear relationship visible to the naked eye, the plots allow us to see the range of cyclist counts across different levels of proximity to these facilities.

# 04 Multivariate Spatial Analysis

We perform a multiple regression to see how spatial features collectively influence the traffic volume.

In [ ]:
import statsmodels.api as sm

# Define predictors (X) and target (y)
spatial_features = ['park_count', 'school_count', 'station_count', 'dist_nearest_station', 'dist_nearest_school']
X = df[spatial_features]
y = df['count']

# Add a constant for the intercept
X = sm.add_constant(X)

# Fit the model
model = sm.OLS(y, X).fit()

# Display summary
print(model.summary())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlation matrix for spatial features
spatial_features = ['park_count', 'school_count', 'station_count', 'dist_nearest_station', 'dist_nearest_school']
corr_matrix = df[spatial_features].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap: Spatial Features', fontsize=15)
plt.show()

Moderate correlation (0.40) between school_count and park_count.
Negative correlation (-0.51) between school_count and dist_nearest_school, which is expected since more schools in a radius typically means a shorter distance to the nearest one.
Most other correlations are relatively low, suggesting that while some multicollinearity exists (as indicated by the regression's condition number), there aren't two variables that are perfectly redundant.

### Summary of Spatial and Multivariate Analysis

*   **Municipality Leaders**: Hasselt, Leuven, and Zemst exhibit the highest average cyclist traffic in the 2024 dataset.
*   **Spatial Correlations**: There is a weak but statistically significant positive correlation between proximity to green spaces (parks) and cyclist volume.
*   **Regression Insights**:
    *   Spatial features (parks, schools, stations) explain approximately **7.1%** of the variance in traffic counts.
    *   **Schools and Parks** within a 500m radius are positive predictors of cyclist counts.
    *   The model indicates that while spatial infrastructure is a significant factor, the majority of traffic variation is likely driven by temporal (hour, day) and environmental (weather) variables.

# 05 Weather Analysis

Based on the correlation matrix, we can see how strongly environmental factors like temperature or solar radiation influence cycling volume compared to the spatial features analyzed earlier.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the non-spatial features of interest
weather_temporal_features = [
    'count', 'temperature_2m', 'relative_humidity_2m',
    'precipitation', 'wind_speed_10m', 'shortwave_radiation',
    'hour', 'month', 'day_of_week'
]

# Calculate correlation matrix
non_spatial_corr = df[weather_temporal_features].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(non_spatial_corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Heatmap: Weather, Time, and Cyclist Count', fontsize=15)
plt.show()

### Key Findings: Weather & Temporal Correlations

The correlation heatmap reveals several critical drivers of cycling volume in 2024:

*   **Hour of Day (0.42)**: This is the strongest predictor, reflecting the dominant influence of daily commuting and human activity cycles.
*   **Solar Radiation (0.24) & Temperature (0.14)**: Both show clear positive correlations, confirming a preference for warm, sunny conditions.
*   **Relative Humidity (-0.16)**: Shows a negative correlation, as higher humidity (often linked to rain or discomfort) tends to reduce cycling activity.
*   **Precipitation**: While showing a weak negative linear correlation, its impact is significant; the low coefficient is primarily due to the high frequency of zero-precipitation hours.

Overall, these dynamic environmental and temporal factors have a more pronounced linear relationship with traffic volume than static spatial infrastructure features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up a multi-plot figure for weather variables
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Temperature vs Count with a regression trend
sns.regplot(data=df.sample(20000, random_state=42), x='temperature_2m', y='count',
            scatter_kws={'alpha':0.1, 's':10}, line_kws={'color':'red'}, ax=axes[0])
axes[0].set_title('Impact of Temperature on Cyclist Counts (Sampled Data)', fontsize=14)
axes[0].set_xlabel('Temperature (2m) [°C]')
axes[0].set_ylabel('Cyclist Count')

# 2. Shortwave Radiation vs Count
sns.regplot(data=df.sample(20000, random_state=42), x='shortwave_radiation', y='count',
            scatter_kws={'alpha':0.1, 's':10, 'color':'orange'}, line_kws={'color':'blue'}, ax=axes[1])
axes[1].set_title('Impact of Solar Radiation on Cyclist Counts (Sampled Data)', fontsize=14)
axes[1].set_xlabel('Shortwave Radiation [W/m²]')
axes[1].set_ylabel('Cyclist Count')

plt.tight_layout()
plt.show()

# Calculate average counts by temperature bins for a clearer trend
# Setting observed=False explicitly to handle the FutureWarning
df['temp_bin'] = pd.cut(df['temperature_2m'], bins=range(-10, 45, 5))
temp_trend = df.groupby('temp_bin', observed=False)['count'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=temp_trend, x='temp_bin', y='count', palette='coolwarm', hue='temp_bin', legend=False)
plt.title('Average Cyclist Count by Temperature Intervals', fontsize=14)
plt.xlabel('Temperature Range (°C)')
plt.ylabel('Average Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Detailed Weather Observations

Based on the visual analysis of environmental factors:

*   **Temperature & Solar Radiation**: Both scatter plots show a clear upward trend, confirming that as it gets warmer and sunnier, cyclist volume increases significantly.
*   **Optimal Temperature Range**: The bar chart reveals that cycling activity peaks when temperatures are between **25°C and 30°C**. Interestingly, there is a slight dip in the very highest bin (30-35°C), suggesting a threshold where extreme heat may begin to discourage cycling.
*   **Cold Weather Impact**: Activity is at its lowest when temperatures are below 0°C, remaining relatively flat until it crosses the 5-10°C threshold.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(50000, random_state=42), x='precipitation', y='count', alpha=0.2, color='royalblue')

plt.title('Relationship Between Precipitation and Cyclist Counts (Sampled Data)', fontsize=14)
plt.xlabel('Precipitation (mm)')
plt.ylabel('Cyclist Count')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Create bins for precipitation
df['precip_bin'] = pd.cut(df['precipitation'], bins=[0, 0.1, 1, 5, 10, 50], labels=['0-0.1', '0.1-1', '1-5', '5-10', '10+'], include_lowest=True)

# Calculate average counts per precipitation level
precip_impact = df.groupby('precip_bin', observed=False)['count'].mean().reset_index()

# Display the results
print("Average Cyclist Count by Precipitation Level (mm):")
display(precip_impact)

# Visualize the results
plt.figure(figsize=(10, 6))
sns.barplot(data=precip_impact, x='precip_bin', y='count', palette='Blues_r', hue='precip_bin', legend=False)
plt.title('Average Cyclist Count vs. Precipitation Intensity', fontsize=14)
plt.xlabel('Precipitation Range (mm)')
plt.ylabel('Average Cyclist Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Group by weekend status and precipitation bins
precip_weekend_impact = df.groupby(['is_weekend', 'precip_bin'], observed=False)['count'].mean().reset_index()

# Visualize the comparison
plt.figure(figsize=(12, 6))
sns.barplot(data=precip_weekend_impact, x='precip_bin', y='count', hue='is_weekend', palette='Paired')

plt.title('Precipitation Impact: Weekdays vs. Weekends', fontsize=14)
plt.xlabel('Precipitation Range (mm)')
plt.ylabel('Average Cyclist Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Day Type')
plt.show()

# Display the underlying data
display(precip_weekend_impact.pivot(index='precip_bin', columns='is_weekend', values='count'))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Regression plot to see general trend
plt.figure(figsize=(12, 6))
sns.regplot(data=df.sample(20000, random_state=42), x='wind_speed_10m', y='count',
            scatter_kws={'alpha':0.1, 's':10, 'color':'gray'}, line_kws={'color':'red'})
plt.title('Cyclist Count vs. Wind Speed (Sampled Data)', fontsize=14)
plt.xlabel('Wind Speed at 10m (km/h)')
plt.ylabel('Cyclist Count')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 2. Binned analysis for average impact
df['wind_bin'] = pd.cut(df['wind_speed_10m'], bins=range(0, 50, 5))
wind_impact = df.groupby('wind_bin', observed=False)['count'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=wind_impact, x='wind_bin', y='count', palette='GnBu_r', hue='wind_bin', legend=False)
plt.title('Average Cyclist Count by Wind Speed Intervals', fontsize=14)
plt.xlabel('Wind Speed Range (km/h)')
plt.ylabel('Average Cyclist Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Group by weekend status and wind bins
wind_weekend_impact = df.groupby(['is_weekend', 'wind_bin'], observed=False)['count'].mean().reset_index()

# Visualize the comparison
plt.figure(figsize=(14, 7))
sns.barplot(data=wind_weekend_impact, x='wind_bin', y='count', hue='is_weekend', palette='coolwarm')

plt.title('Impact of Wind Speed: Weekdays vs. Weekends', fontsize=15)
plt.xlabel('Wind Speed Range (km/h)', fontsize=12)
plt.ylabel('Average Cyclist Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(title='Day Type')
plt.show()

# Display the table for precise comparison
display(wind_weekend_impact.pivot(index='wind_bin', columns='is_weekend', values='count'))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Solar Radiation Analysis
plt.figure(figsize=(12, 6))
sns.lineplot(data=df.sample(10000, random_state=42), x='sunshine_duration', y='count', color='orange')
plt.title('Cyclist Count vs. Sunshine Duration', fontsize=14)
plt.xlabel('Sunshine Duration (seconds)')
plt.ylabel('Average Cyclist Count')
plt.grid(True, alpha=0.3)
plt.show()

# 2. Humidity Binned Analysis
df['humidity_bin'] = pd.cut(df['relative_humidity_2m'], bins=range(0, 110, 10))
humidity_impact = df.groupby('humidity_bin', observed=False)['count'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=humidity_impact, x='humidity_bin', y='count', hue='humidity_bin', palette='YlGnBu', legend=False)
plt.title('Impact of Relative Humidity on Cycling Activity', fontsize=14)
plt.xlabel('Humidity Range (%)')
plt.ylabel('Average Count')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Focus on top municipalities (gemeente) by volume for meaningful correlation
top_gemeentes = df.groupby('gemeente')['count'].mean().nlargest(10).index
subset_df = df[df['gemeente'].isin(top_gemeentes)]

def get_weather_corrs(group):
    return pd.Series({
        'Temp Correlation': group['count'].corr(group['temperature_2m']),
        'Solar Correlation': group['count'].corr(group['shortwave_radiation']),
        'Rain Correlation': group['count'].corr(group['precipitation'])
    })

regional_weather_impact = subset_df.groupby('gemeente').apply(get_weather_corrs, include_groups=False).reset_index()

# Visualize the differences
plt.figure(figsize=(14, 8))
regional_melted = regional_weather_impact.melt(id_vars='gemeente', var_name='Weather Metric', value_name='Correlation')
sns.barplot(data=regional_melted, y='gemeente', x='Correlation', hue='Weather Metric', palette='coolwarm')

plt.title('Weather Sensitivity Comparison Across Top Municipalities', fontsize=15)
plt.xlabel('Pearson Correlation Coefficient', fontsize=12)
plt.ylabel('Municipality', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

display(regional_weather_impact)


We compare the correlation coefficients between weather metrics and cyclist counts, split by day type.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Reuse the top_gemeentes defined previously (Top 10 by average count)
top_gemeentes = df.groupby('gemeente')['count'].mean().nlargest(10).index
subset_df = df[df['gemeente'].isin(top_gemeentes)]

def get_split_weather_corrs(group):
    return pd.Series({
        'Temp (Weekday)': group[group['is_weekend'] == 'Weekday']['count'].corr(group[group['is_weekend'] == 'Weekday']['temperature_2m']),
        'Temp (Weekend)': group[group['is_weekend'] == 'Weekend']['count'].corr(group[group['is_weekend'] == 'Weekend']['temperature_2m']),
        'Solar (Weekday)': group[group['is_weekend'] == 'Weekday']['count'].corr(group[group['is_weekend'] == 'Weekday']['shortwave_radiation']),
        'Solar (Weekend)': group[group['is_weekend'] == 'Weekend']['count'].corr(group[group['is_weekend'] == 'Weekend']['shortwave_radiation'])
    })

# Grouping by 'gemeente' instead of 'district'
split_impact = subset_df.groupby('gemeente').apply(get_split_weather_corrs, include_groups=False).reset_index()

# Reshape for visualization
split_melted = split_impact.melt(id_vars='gemeente', var_name='Metric_Type', value_name='Correlation')
split_melted[['Metric', 'Day Type']] = split_melted['Metric_Type'].str.extract(r'(.*) \((.*)\)')

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

sns.barplot(data=split_melted[split_melted['Metric'] == 'Temp'], y='gemeente', x='Correlation', hue='Day Type', ax=ax1, palette='coolwarm')
ax1.set_title('Temperature Sensitivity by Municipality: Weekday vs Weekend')

sns.barplot(data=split_melted[split_melted['Metric'] == 'Solar'], y='gemeente', x='Correlation', hue='Day Type', ax=ax2, palette='YlOrBr')
ax2.set_title('Solar Radiation Sensitivity by Municipality: Weekday vs Weekend')

plt.tight_layout()
plt.show()

display(split_impact)

### Comparison of Weather Sensitivity Across Top 10 Municipalities

Comparing the weather impact metrics reveals distinct cycling behaviors across regions:

*   **High Weekend Sensitivity**: In municipalities like **Zemst** and **Aarschot**, the correlation with solar radiation surges to **0.67–0.74** on weekends. This indicates that cycling in these areas is highly weather-dependent and likely driven by recreational purposes.
*   **Coastal Characteristics**: **Nieuwpoort** stands out with consistently high correlations for both temperature and solar radiation (above 0.43) across both weekdays and weekends. This suggests a tourism-driven pattern where cycling activity remains weather-sensitive throughout the entire week.
*   **Commuting Resilience**: Conversely, **Hasselt** demonstrates higher "commuting resilience." Its weather sensitivity on weekends is actually lower than on weekdays, suggesting that cycling in this city is driven more by necessity/utility, making it less susceptible to fluctuations in weather conditions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter for records where there was actual snowfall
non_zero_snow = df[df['snowfall'] > 0].copy()

print(f"Total records with snowfall: {len(non_zero_snow)}")

if len(non_zero_snow) > 0:
    # 2. Visualize the relationship between snowfall intensity and count
    plt.figure(figsize=(12, 6))
    sns.scatterplot(data=non_zero_snow, x='snowfall', y='count', alpha=0.5, color='lightskyblue')
    plt.title('Impact of Snowfall Intensity on Cyclist Counts (Snowy Hours Only)', fontsize=14)
    plt.xlabel('Snowfall (cm)')
    plt.ylabel('Cyclist Count')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    # 3. Compare Average Counts: Snow vs No Snow
    df['has_snow'] = df['snowfall'] > 0
    snow_comparison = df.groupby('has_snow')['count'].mean().reset_index()
    snow_comparison['has_snow'] = snow_comparison['has_snow'].map({True: 'Snowing', False: 'No Snow'})

    plt.figure(figsize=(8, 6))
    sns.barplot(data=snow_comparison, x='has_snow', y='count', hue='has_snow', palette='Blues', legend=False)
    plt.title('Average Cyclist Count: Snowing vs. Clear Weather', fontsize=14)
    plt.ylabel('Average Count')
    plt.show()

    display(snow_comparison)
else:
    print("No significant snowfall recorded in the 2024 dataset.")

### Impact of Snowfall on Cycling Volume

Based on the analysis of nearly 14,000 records with active snowfall in 2024:

*   **Significant Volume Reduction**: The average cyclist count drops from **10.66** (clear weather) to **5.74** (snowing), representing a **46% decrease** in activity.
*   **Intensity Threshold**: High-volume traffic disappears almost entirely during snowfall. The scatter plot indicates that once snowfall exceeds even a minimal intensity, counts are consistently restricted to low levels, suggesting that only essential/utility cycling remains.
*   **Commuting vs. Recreational**: Snowfall typically occurs in winter months (the lowest overall period), and its deterrent effect is strong regardless of the time of day.

### Impact of Weather on Cycling Volume (2024)

| Weather Factor | Optimal Range / Threshold | Nature of Impact | Observation |
| :--- | :--- | :--- | :--- |
| **Temperature** | 25-30°C | Positive (+0.14) | Higher activity in warm weather; sharp decline below 5°C. |
| **Solar Radiation** | > 600 W/m² | Strong Positive (+0.24) | Strongest driver; significant boost for weekend recreational cycling. |
| **Snowfall** | > 0 cm | Strong Negative | Reduces average cyclist volume by 46% (10.66 → 5.74). |
| **Precipitation** | > 1.0 mm | Negative | Significant reduction in volume; declines as rain intensity increases. |
| **Wind Speed** | > 15 km/h | Negative | Increased physical resistance leads to lower average counts. |

###  Summary: Impact of Weather on Cycling Volume (2024)

Based on our data analysis, here are the primary ways environmental factors influence cyclist counts:

1.  **Temperature (The Warmth Factor):** There is a clear positive relationship (+0.14 correlation). The 'sweet spot' for cycling is between **25°C and 30°C**. Once the temperature drops below 5°C, we see a sharp decline in activity.
2.  **Solar Radiation (The Sunlight Driver):** This is the most influential environmental predictor (+0.24 correlation). High solar intensity (> 600 W/m²) significantly boosts traffic, especially for recreational weekend cycling.
3.  **Snowfall (The Major Disruptor):** Snow has the most drastic negative impact, cutting average cyclist volume nearly in half (**-46%**), as users switch to other transport modes or stay home.
4.  **Precipitation (The Rain Deterrent):** Active rain (over 1.0 mm) consistently lowers counts. Although the linear correlation seems low (-0.04), this is mainly because it doesn't rain most of the time; when it does, the impact is immediate.
5.  **Wind Speed (The Resistance Factor):** As wind speeds climb above 15 km/h, average counts begin to taper off due to the increased effort required to cycle.

# 06 Predictive Modeling: Linear Regression Baseline

In this section, we build a baseline model to predict cyclist counts using the features identified during our EDA.

Detailed Statistical Analysis of Model Features

In this section, we analyze the variance and statistical significance (p-values) of the features used in our predictive model.

In [ ]:
import statsmodels.api as sm
import pandas as pd

# 1. Prepare features for the comprehensive model
# We include Time, Weather, and Spatial features
features_for_regression = [
    'hour', 'temperature_2m', 'shortwave_radiation',
    'precipitation', 'wind_speed_10m', 'relative_humidity_2m',
    'park_count', 'school_count', 'station_count'
]

X_full = df[features_for_regression].copy()

# Add categorical flag for weekend
X_full['is_weekend'] = (df['is_weekend'] == 'Weekend').astype(int)

# Target variable
y = df['count']

# Add constant for intercept
X_full = sm.add_constant(X_full)

# 2. Fit the OLS model
model_full = sm.OLS(y, X_full).fit()

# 3. Display the summary to identify the most impactful factors
print(model_full.summary())

In [ ]:
# Extract and sort standardized coefficients to compare impact magnitude
# Note: Since features have different scales, we'll look at the t-values for relative importance
importance_df = pd.DataFrame({
    'Factor': model_full.params.index,
    'Coefficient': model_full.params.values,
    't-value': model_full.tvalues.values,
    'p-value': model_full.pvalues.values
}).sort_values('t-value', ascending=False)

print("\n### Factors Ranked by Statistical Influence (t-value) ###")
display(importance_df)

### Evaluation and Analysis of Regression Results

Based on the statistical output from the OLS model, here is a comprehensive evaluation of the findings:

#### 1. Model Performance (R-squared: 0.131)
*   **Assessment**: An R-squared of 0.131 indicates that the model explains approximately **13.1%** of the variance in cyclist counts. While this is a modest start for a baseline linear model, it suggests that traffic patterns are influenced by complex, non-linear relationships or additional factors not captured in this simple model.

#### 2. Statistical Significance (p-values)
*   **Assessment**: All features in the model have **p-values of 0.000**, which is well below the standard 0.05 threshold. This means every variable selected—such as hour, temperature, school count, and weekend status—has a statistically significant impact on predicting cyclist traffic.

#### 3. Key Feature Drivers (Coefficients)
*   **Positive Drivers**:
    *   **Infrastructure**: `school_count` (1.77) and `park_count` (1.54) are strong positive predictors.
    *   **Environment**: `temperature_2m` (0.05) and `shortwave_radiation` (0.03) show positive trends. Although coefficients seem small, the large scale of radiation makes it a powerful driver.
*   **Negative Drivers**:
    *   **Temporal**: `is_weekend` (-3.72) shows that weekend traffic is significantly lower, reinforcing the "utility/commuter" nature of this dataset.
    *   **Weather**: `precipitation` (-1.03) indicates a direct drop in cycling activity for every mm of rain.

#### 4. Multicollinearity Warning
*   **Note**: The large **Condition Number (2.83e+03)** suggests potential multicollinearity, likely between temperature and solar radiation. While the predictive trend holds, caution is advised when interpreting exact individual magnitudes.

#### 5. Next Steps
The baseline model confirms our significant features but highlights the limitations of linear assumptions. Moving to non-linear algorithms like **Random Forest** or **XGBoost** should be the next priority to improve the R-squared and better capture the synergy between weather and location.